In [ ]:
import sys
# This forces the installation into the exact Python kernel VS Code is using right now
!{sys.executable} -m pip install playwright pandas lxml
!{sys.executable} -m playwright install chromium

  Using cached playwright-1.58.0-py3-none-macosx_11_0_arm64.whl.metadata (3.5 kB)
  Using cached pyee-13.0.1-py3-none-any.whl.metadata (3.0 kB)
Using cached playwright-1.58.0-py3-none-macosx_11_0_arm64.whl (41.0 MB)
Using cached pyee-13.0.1-py3-none-any.whl (15 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [playwright]3 [playwright]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [9]:
import sys
# This forces the installation into the exact Python kernel VS Code is using right now
!{sys.executable} -m pip install playwright pandas lxml
!{sys.executable} -m playwright install chromium


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [1]:
!pip install selenium webdriver-manager pandas

In [1]:
import pandas as pd
import asyncio
import re
from io import StringIO
from playwright.async_api import async_playwright

async def scrape_district_summaries():
    # The 4 districts you need
    target_districts = ["BATHINDA", "PATIALA", "RUPNAGAR", "FATEHGARH SAHIB"]
    
    async with async_playwright() as p:
        # headless=False so you can see it working
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()

        for district in target_districts:
            print(f"\n🚀 FETCHING SUMMARY FOR: {district}")
            
            try:
                # 1. Start at the main page
                await page.goto("https://pmagy.gov.in/Average-Village-Score", timeout=90000)
                await asyncio.sleep(5)
                
                # 2. Click Punjab
                await page.get_by_text("PUNJAB", exact=True).first.click()
                await asyncio.sleep(5)
                
                # 3. Click the District (Handling Bathinda/Bhatinda and Rupnagar/Ropar)
                d_regex = "^(BATHINDA|BHATINDA)$" if district == "BATHINDA" else \
                          "^(RUPNAGAR|ROPAR)$" if district == "RUPNAGAR" else f"^{district}$"
                
                await page.locator("td").filter(has_text=re.compile(d_regex, re.IGNORECASE)).first.click()
                await asyncio.sleep(5)
                
                # 4. Expand to 100 entries so we get all Blocks in one table
                try:
                    await page.locator("select").first.select_option("100")
                    await asyncio.sleep(3)
                except: pass

                # 5. Grab the table right here and save it
                html = await page.content()
                dfs = pd.read_html(StringIO(html))
                
                # Usually, the biggest table is the one with the Block scores
                df = max(dfs, key=len)
                
                # Clean up and add a column for the district name
                df.dropna(how='all', axis=1, inplace=True)
                df['District_Name'] = district
                
                # Save this specific district to its own CSV
                filename = f"{district.lower().replace(' ', '_')}_summary.csv"
                df.to_csv(filename, index=False)
                print(f"✅ Table for {district} fetched and saved to {filename}")

            except Exception as e:
                print(f"❌ Failed to fetch {district}: {e}")

        await browser.close()
        print("\n🏁 Process complete. All 4 district summary tables are ready.")

# Run the summary fetcher
await scrape_district_summaries()


🚀 FETCHING SUMMARY FOR: BATHINDA
✅ Table for BATHINDA fetched and saved to bathinda_summary.csv

🚀 FETCHING SUMMARY FOR: PATIALA
✅ Table for PATIALA fetched and saved to patiala_summary.csv

🚀 FETCHING SUMMARY FOR: RUPNAGAR
✅ Table for RUPNAGAR fetched and saved to rupnagar_summary.csv

🚀 FETCHING SUMMARY FOR: FATEHGARH SAHIB
✅ Table for FATEHGARH SAHIB fetched and saved to fatehgarh_sahib_summary.csv

🏁 Process complete. All 4 district summary tables are ready.


In [2]:
import glob
files = glob.glob("*_summary.csv")
final_df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
final_df.to_csv("final_punjab_districts_summary.csv", index=False)
print("🚀 Master summary dataset created!")

🚀 Master summary dataset created!
